# Exploration et datacard : parties Diamant de League of Legends à 15 minutes

**Jeu de données** : [OpenML 43635, League-of-Legends-Diamond-Games-(First-15-Minutes)](https://www.openml.org/search?type=data&status=active&id=43635)
**Auteur** : Louis, EPSI (2026-2027)

| Étape | Consigne | Section |
|---|---|---|
| 1 | Trouver une source OpenML bien documentée, dont la collecte est décrite | 1 |
| 2 | Importer les données et les explorer | 2 à 7 |
| 3 | Dictionnaire des données : nom, type, intervalle, moyenne, médiane, quartiles, répartition des classes, z-scores élevés, valeurs manquantes, doublons | 8 (détails en 4 et 7) |
| 4 | Deux projets de ML réalistes et un projet difficile | 9 |
| 5 | Datacard : entête rédigée à la main et partie données écrite par un script | 10 |

**Plan**

0. Préparation
1. Source des données et collecte
2. Modèle du domaine (ce que l'on s'attend à trouver)
3. Import des données
4. Premier aperçu et qualité : structure, valeurs manquantes, doublons, types
5. Confrontation des données avec le modèle du domaine
6. Nettoyage minimal
7. Exploration : statistiques, distributions, classes, valeurs aberrantes, liens avec la cible
8. Dictionnaire des données
9. Projets de Machine Learning
10. Datacard
11. Bilan

**Fichiers du projet** (même dossier que ce notebook) : `generate_datacard.py` (script), `datacard_config.json` et `datacard_header.md` (parties rédigées à la main), `DATACARD.md` (datacard générée).

## 0. Préparation

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, MultipleLocator
from IPython.display import Markdown, display

import generate_datacard as gdc  # script de la datacard, réutilisé ici pour ne pas dupliquer les calculs
from clean_data import read_raw   # même lecture du fichier brut que la chaîne de traitement
from common import RAW_ARFF, RAW_METADATA

OPENML_ID = 43635
RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"
for folder in (RAW_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

config = gdc.load_config(PROJECT_DIR / "datacard_config.json")  # rôles, descriptions, règles métier
fr = gdc.fmt_number  # affichage des nombres à la française : 26 349,16

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 2)

# Couleurs : bleu et rouge pour les deux équipes (paire vérifiée lisible par les personnes daltoniennes)
BLUE, RED, VIOLET, NEUTRAL = "#2a78d6", "#e34948", "#4a3aa7", "#898781"
INK, INK_SOFT, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 9.5,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK,
    "axes.titlelocation": "left", "axes.labelcolor": INK_SOFT,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK_SOFT, "ytick.labelcolor": INK_SOFT,
    "legend.frameon": False,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
})
thousands = FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " "))

## 1. Source des données et collecte

**Pourquoi ce jeu de données ?** Il est publié sur OpenML sous licence libre (CC0) et sa description
indique comment les données ont été collectées : API officielle de Riot Games, saison, serveur, rang.
C'est aussi un domaine que je connais en tant que joueur, ce qui aide à juger si les valeurs ont du sens.

Les métadonnées officielles viennent de l'API REST d'OpenML. Le script `collect_data.py` les enregistre
dans `data/raw/` avec le fichier brut : ce sont les données d'entrée du dépôt, le notebook fonctionne donc
hors ligne.

In [ ]:
if not RAW_METADATA.exists():
    raise FileNotFoundError("Métadonnées absentes : lancer « python collect_data.py » depuis la racine du dépôt.")
meta = json.loads(RAW_METADATA.read_text(encoding="utf-8"))

fields = {
    "name": "Nom", "version": "Version", "format": "Format", "licence": "Licence",
    "collection_date": "Date de collecte", "upload_date": "Mise en ligne sur OpenML",
    "md5_checksum": "Somme MD5 du fichier ARFF",
}
display(pd.DataFrame({"valeur": [meta.get(key) for key in fields]}, index=list(fields.values())))
print(meta["description"])

**Ce que la description dit de la collecte**

| Élément | Valeur |
|---|---|
| Méthode | Extraction automatisée (scraping) via l'API officielle de Riot Games, avec l'accord de Riot |
| Saison | « Season 13 » (identifiant de saison renvoyé par l'API) |
| Serveur | NA1 (Amérique du Nord) |
| Rang | Diamant, divisions I, II, III et IV |
| Fenêtre observée | Les 15 premières minutes de chaque partie |
| Volume annoncé | Environ 50 000 parties |
| Date de collecte | 31/07/2020 |
| Origine | Jeu Kaggle de Ben Fattori, inspiré du jeu « League of Legends Diamond Ranked Games (10 min) » |

**Points d'attention dès la lecture**

- « Season 13 » n'est pas une année : c'est l'identifiant de saison de l'API. La table de référence publiée
  par Riot (`seasons.json`) s'arrête à cet identifiant, libellé « SEASON 2019 », alors que des saisons ont
  suivi. La période exacte des parties est donc inconnue ; on sait seulement qu'elles précèdent la collecte.
- Ni la version du jeu (patch) ni la façon dont les parties ont été choisies ne sont précisées.
- L'import sur OpenML a fait perdre le lien vers le jeu d'origine (« Inspired by the following dataset , ... »).
- L'auteur annonce un plafond d'environ 70 % d'exactitude sur le jeu à 10 minutes : c'est un repère
  pour juger les modèles de la section 9.

## 2. Modèle du domaine (avant de regarder les chiffres)

Avant d'explorer, on formalise ce que l'on **s'attend** à trouver, d'après les règles du jeu en 2020.
On confrontera ensuite les données à ces attentes (section 5) : un écart signale soit une erreur de
collecte, soit une mauvaise compréhension d'une colonne.

**Entités**

```
Partie (matchId), issue : blue_win (1 = victoire de l'équipe bleue)
 ├── Équipe bleue (5 joueurs), état cumulé à 15:00
 │    ├── Ressources : Gold, MinionsKilled, JungleMinionsKilled, AvgLevel
 │    └── Actions    : ChampKills, HeraldKills, DragonKills, TowersDestroyed
 └── Équipe rouge (5 joueurs), mêmes indicateurs avec le préfixe red
```

Les ressources sont des **états** (ce que l'équipe possède à 15:00), les actions des **événements**
comptés depuis le début. Les deux équipes étant symétriques, ce sont surtout les **écarts** entre elles
qui devraient expliquer la victoire.

**Attentes par concept**

| Concept | Colonnes | Ce que l'on attend (règles de 2020) |
|---|---|---|
| Issue | `blue_win` | 0 ou 1, environ 50 % de victoires bleues |
| Partie | `matchId` | Identifiant entier, unique |
| Or | `blueGold`, `redGold` | Au moins 2 500 (5 joueurs × 500 pièces de départ), de l'ordre de 25 000 à 15 minutes |
| Sbires, jungle | `*MinionsKilled`, `*JungleMinionsKilled` | Entiers positifs |
| Niveau moyen | `*AvgLevel` | Entre 1 et 18, multiple de 0,2 (moyenne de 5 niveaux entiers) |
| Éliminations | `*ChampKills` | Entiers positifs |
| Dragons | `*DragonKills` | Premier dragon à 5:00, retour 5 minutes après sa mort : au plus 2 par partie avant 15:00, presque toujours au moins 1 en Diamant |
| Héraut de la Faille | `*HeraldKills` | Apparition à 8:00, retour 6 minutes après sa mort : au plus 2 par partie avant 15:00 |
| Tours | `*TowersDestroyed` | 11 tours par équipe, donc entre 0 et 11 ; détruire des tours devrait aller avec la victoire |

**Hypothèses à vérifier**

- **H1** : chaque partie n'apparaît qu'une fois.
- **H2** : toutes les valeurs respectent les bornes ci-dessus.
- **H3** : les colonnes « dragons » et « Héraut » comptent ce que leur nom indique.
- **H4** : `*TowersDestroyed` compte les tours détruites **par** l'équipe.
- **H5** : il n'y a pas de partie anormale (joueur absent, partie écourtée).

Ces attentes sont aussi écrites sous forme de règles dans `datacard_config.json`, pour être vérifiées
automatiquement à chaque nouveau batch.

## 3. Import des données

Le fichier brut publié par OpenML (format ARFF) est versionné dans `data/raw/` : ce sont les données
d'entrée du projet. `collect_data.py` vérifie son intégrité (somme MD5 publiée par OpenML) et le
retélécharge si besoin. Chaque nouveau batch aura son propre fichier dans `data/raw/`, ce qui permet de
régénérer la datacard et de comparer les versions.

In [ ]:
if not RAW_ARFF.exists():
    raise FileNotFoundError("Fichier brut absent : lancer « python collect_data.py » depuis la racine du dépôt.")
raw = read_raw(RAW_ARFF)
print(f"Lu : {RAW_ARFF.relative_to(PROJECT_DIR).as_posix()}")

print(f"{fr(raw.shape[0])} lignes x {raw.shape[1]} colonnes")
raw.head()

## 4. Premier aperçu et qualité des données

### 4.1 Structure et types

In [ ]:
overview = pd.DataFrame({
    "type": raw.dtypes.astype(str),
    "valeurs distinctes": raw.nunique(),
    "manquants": raw.isna().sum(),
    "première ligne": raw.iloc[0].astype(object),
})
overview

- **48 651 lignes, 19 colonnes**, cohérent avec les « environ 50 000 parties » annoncées.
- **Tout est stocké en `float64`** : le format ARFF ne distingue pas entiers et décimaux. `matchId`
  s'affiche même en notation scientifique, symptôme d'un identifiant stocké comme un nombre à virgule.
- **`Unnamed:_0`** a autant de valeurs distinctes que de lignes : c'est l'index d'un DataFrame pandas
  exporté avec les données (nom typique d'un `to_csv` sans `index=False`). Il n'a aucun sens métier.
- **`blueDragonKills` et `redDragonKills` n'ont qu'une seule valeur** : à examiner en section 5.

### 4.2 Valeurs manquantes

In [ ]:
missing = raw.isna().sum()
print(f"Valeurs manquantes : {missing.sum()} au total, {(missing > 0).sum()} colonne(s) concernée(s)")

Aucune valeur manquante : pas d'imputation à prévoir.

### 4.3 Doublons

In [ ]:
technical_cols = config["technical_columns"]  # ['Unnamed:_0']
print(f"Lignes identiques sur toutes les colonnes       : {raw.duplicated().sum()}")
print(f"Lignes identiques en ignorant {technical_cols[0]}    : {raw.drop(columns=technical_cols).duplicated().sum()}")
print(f"matchId présents plusieurs fois                 : {raw['matchId'].duplicated().sum()}")

dup_rows = raw[raw["matchId"].duplicated(keep=False)].sort_values(["matchId", "Unnamed:_0"])
gaps = dup_rows.groupby("matchId")["Unnamed:_0"].agg(lambda s: s.max() - s.min())
print(f"Copies d'une même partie sur deux lignes consécutives : {(gaps == 1).sum()} cas sur {len(gaps)}")
dup_rows.head(6)

**Le piège** : `raw.duplicated()` ne trouve aucun doublon, car `Unnamed:_0` diffère entre deux copies
d'une même partie. En ignorant cette colonne, on trouve **19 parties en double**, avec exactement les
mêmes valeurs : la même partie a été collectée deux fois (le plus souvent sur deux lignes consécutives).
Hypothèse H1 rejetée ; les doublons seront supprimés.

### 4.4 Types de données

In [ ]:
integer_like = [c for c in raw.columns if (raw[c] % 1 == 0).all()]
print(f"{len(integer_like)} colonnes sur {raw.shape[1]} ne contiennent que des valeurs entières.")
print("Colonnes réellement décimales :", [c for c in raw.columns if c not in integer_like])

# Pourquoi des entiers SIGNÉS : la copie Parquet d'OpenML stocke les comptages en uint8 (non signé)
print("uint8 : 6 - 12 =", (pd.Series([6], dtype="uint8") - pd.Series([12], dtype="uint8")).iloc[0])
print("int64 : 6 - 12 =", (pd.Series([6], dtype="int64") - pd.Series([12], dtype="int64")).iloc[0])

Seuls les niveaux moyens sont de vrais décimaux. On convertira les autres colonnes en entiers
**signés** (`int64`) : avec un type non signé comme le `uint8` de la copie Parquet d'OpenML, un écart
de 6 - 12 éliminations vaudrait 250 au lieu de -6, et tous les écarts entre équipes seraient faux
sans aucun message d'erreur.

## 5. Confrontation des données avec le modèle du domaine

### 5.1 Règles métier

Les attentes de la section 2 sont déclarées dans `datacard_config.json` sous forme d'expressions pandas
qui décrivent une ligne **valide**. La fonction `evaluate_rules` est celle du script de la datacard :
le notebook et la datacard appliquent exactement les mêmes contrôles.

In [ ]:
rules = gdc.evaluate_rules(raw, config)
rules[["regle", "severite", "violations", "pct_violations", "statut"]]

Aucune règle de sévérité « erreur » n'est violée : pas de valeur impossible (H2 vérifiée). Les trois
**alertes** en revanche sont parlantes ; on les examine une par une.

### 5.2 Dragons

In [ ]:
raw[["blueDragonKills", "redDragonKills"]].value_counts()

**Aucun dragon tué avant 15 minutes, dans aucune partie.** C'est impossible en Diamant, où le premier
dragon apparaît à 5:00 et est presque toujours pris avant 15:00. La colonne est cassée, très
probablement par une erreur dans le script de collecte. Hypothèse H3 rejetée pour les dragons :
les deux colonnes seront supprimées.

### 5.3 Héraut de la Faille

In [ ]:
epic_total = (raw["blueHeraldKills"] + raw["redHeraldKills"]).astype(int)
herald_check = epic_total.value_counts().sort_index().rename("parties").to_frame()
herald_check["part (%)"] = (herald_check["parties"] / len(raw) * 100).round(1)
herald_check.index.name = "total bleu + rouge"
print(f"Moyenne par partie : {epic_total.mean():.2f}")
herald_check

Avec les règles de 2020, une partie ne peut pas compter **plus de 2 Hérauts** avant 15:00. Or, d'après
cette colonne, **65,6 % des parties en comptent 3 ou 4**.

L'hypothèse « dragons + Hérauts » colle en revanche parfaitement :

- maximum théorique : 2 dragons + 2 Hérauts = **4**, exactement le maximum observé ;
- partie typique : 2 dragons + 1 Héraut = **3**, la valeur la plus fréquente ;
- moyenne observée de **2,6** monstres épiques par partie, plausible en Diamant.

Cela expliquerait aussi pourquoi les colonnes dragons sont vides : les dragons auraient été comptés
ici. Conclusion : `*HeraldKills` compte très probablement **tous les monstres épiques**. On la renomme
`*EpicMonsters`, en gardant cette interprétation comme une hypothèse dans la datacard.

### 5.4 Tours

In [ ]:
towers = ["blueTowersDestroyed", "redTowersDestroyed"]
tower_check = pd.DataFrame({
    "corrélation avec blue_win": raw[towers].corrwith(raw["blue_win"]),
    "corrélation avec blueGold": raw[towers].corrwith(raw["blueGold"]),
    "corrélation avec redGold": raw[towers].corrwith(raw["redGold"]),
}).round(2)
display(tower_check)

big = raw[raw["blueTowersDestroyed"] >= 8]
print(f"Parties avec blueTowersDestroyed >= 8 : {len(big)}, dont {int(big['blue_win'].sum())} gagnées par l'équipe bleue")

Si `blueTowersDestroyed` comptait les tours détruites **par** l'équipe bleue, elle irait de pair avec
la victoire bleue et avec l'or bleu (une tour rapporte de l'or à ceux qui la détruisent). C'est
l'inverse : corrélation **négative** avec `blue_win` et `blueGold`, nettement **positive** avec
`redGold`, et l'équipe bleue perd toutes les parties où elle aurait « détruit » 8 tours ou plus.

La colonne compte donc les tours **perdues** par l'équipe bleue (détruites par l'équipe rouge), et
symétriquement pour `redTowersDestroyed`. Explication probable : dans l'API de Riot, l'événement de
destruction d'un bâtiment indique l'équipe **propriétaire** du bâtiment, pas celle qui l'a détruit.
Hypothèse H4 rejetée ; on renomme en `*TowersLost`.

### 5.5 Parties anormales

In [ ]:
low_level = raw[(raw["blueAvgLevel"] < 7) | (raw["redAvgLevel"] < 7)]
low_side_lost = ((low_level["blueAvgLevel"] < low_level["redAvgLevel"]) != (low_level["blue_win"] == 1)).mean()
print(f"{len(low_level)} parties avec un niveau moyen < 7 pour au moins une équipe")
print(f"L'équipe au niveau le plus bas a perdu dans {low_side_lost:.0%} de ces parties")
cols = ["blueAvgLevel", "redAvgLevel", "blueGold", "redGold", "blueChampKills", "redChampKills", "blue_win"]
low_level[cols].sort_values(["redAvgLevel", "blueAvgLevel"]).head(6)

À 15 minutes, un niveau moyen de 2,4 signifie que plusieurs joueurs sont restés quasiment au niveau 1 :
joueurs absents ou déconnectés. Dans les 30 cas, l'équipe en retard de niveau a perdu. Ces parties ne sont pas des erreurs de mesure, mais elles ne
représentent pas une partie normale (H5 rejetée pour une trentaine de parties). On les **garde en les
signalant** (`partie_suspecte`) pour pouvoir les exclure lors de la modélisation.

### 5.6 Bilan des décisions

| Colonne(s) | Constat | Décision |
|---|---|---|
| `Unnamed:_0` | Index pandas exporté par erreur, masque les doublons | Supprimer |
| `matchId` | Stocké en flottant, 19 parties en double | Convertir en entier, supprimer les doublons |
| `*DragonKills` | Toujours 0 | Supprimer |
| `*HeraldKills` | Compte très probablement dragons + Hérauts | Renommer `*EpicMonsters` |
| `*TowersDestroyed` | Compte les tours perdues par l'équipe | Renommer `*TowersLost` |
| `*AvgLevel` < 7 | Une trentaine de parties anormales (joueur absent) | Signaler (`partie_suspecte`) |
| Comptages | Entiers stockés en flottant | Convertir en `int64` (signé) |

Le jeu brut `raw` reste intact : c'est lui que documente la datacard (section 10).

## 6. Nettoyage minimal

In [ ]:
RENAME = {
    "blueHeraldKills": "blueEpicMonsters", "redHeraldKills": "redEpicMonsters",
    "blueTowersDestroyed": "blueTowersLost", "redTowersDestroyed": "redTowersLost",
}
DECIMAL_COLS = ["blueAvgLevel", "redAvgLevel"]

df = (
    raw.drop(columns=["Unnamed:_0", "blueDragonKills", "redDragonKills"])
       .drop_duplicates()
       .rename(columns=RENAME)
       .reset_index(drop=True)
)
int_cols = [c for c in df.columns if c not in DECIMAL_COLS]
df[int_cols] = df[int_cols].astype("int64")
df["partie_suspecte"] = (df["blueAvgLevel"] < 7) | (df["redAvgLevel"] < 7)

TEAM_METRICS = {  # indicateur commun aux deux équipes : libellé lisible
    "Gold": "Or total",
    "MinionsKilled": "Sbires de voie",
    "JungleMinionsKilled": "Monstres de jungle",
    "AvgLevel": "Niveau moyen",
    "ChampKills": "Éliminations",
    "EpicMonsters": "Monstres épiques (ex-HeraldKills)",
    "TowersLost": "Tours perdues (ex-TowersDestroyed)",
}
FEATURES = [f"{team}{metric}" for metric in TEAM_METRICS for team in ("blue", "red")]

print(f"Jeu d'analyse : {fr(len(df))} parties x {df.shape[1]} colonnes "
      f"({fr(len(raw) - len(df))} doublons retirés, {df['partie_suspecte'].sum()} parties signalées)")
df.dtypes.value_counts()

## 7. Exploration

### 7.1 Statistiques descriptives

In [ ]:
df[FEATURES].describe().T

- Les deux équipes ont des profils presque identiques (moyennes et quartiles très proches) : c'est
  attendu, le côté de la carte ne change pas le niveau des joueurs.
- La moitié centrale des équipes (Q1 à Q3) a entre environ 24 400 et 28 100 pièces d'or, 315 à 360
  sbires de voie et 8 à 14 éliminations à 15 minutes : des valeurs cohérentes avec une partie classée.
- Le niveau moyen varie très peu (écart-type de 0,4) : c'est un indicateur serré, où de petits écarts
  peuvent compter.

### 7.2 Distributions

In [ ]:
STEPS = {"Gold": 250, "MinionsKilled": 5, "JungleMinionsKilled": 3, "AvgLevel": 0.2, "ChampKills": 1}
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for ax, (metric, step) in zip(axes.flat, STEPS.items()):
    both = pd.concat([df[f"blue{metric}"], df[f"red{metric}"]])
    lo, hi = both.quantile([0.001, 0.999])
    half = 0.1 if metric == "AvgLevel" else 0.5  # bornes de classes entre deux valeurs possibles
    bins = np.arange(lo - half, hi + step, step)
    for team, color, label in (("blue", BLUE, "Équipe bleue"), ("red", RED, "Équipe rouge")):
        values = df[f"{team}{metric}"]
        ax.hist(values[values.between(lo, hi)], bins=bins, histtype="step", linewidth=1.8,
                color=color, label=label)
    ax.set_title(TEAM_METRICS[metric])
    ax.yaxis.set_major_formatter(thousands)
    if metric == "Gold":
        ax.xaxis.set_major_locator(MultipleLocator(5000))
        ax.xaxis.set_major_formatter(thousands)
for ax in axes[:, 0]:
    ax.set_ylabel("Nombre d'équipes")
legend_ax = axes.flat[-1]
legend_ax.axis("off")
legend_ax.legend(*axes.flat[0].get_legend_handles_labels(), loc="center left", fontsize=10)
legend_ax.text(0.0, 0.25, "Valeurs entre les quantiles 0,1 % et 99,9 %", color=INK_SOFT,
               transform=legend_ax.transAxes)
fig.suptitle("Distribution des indicateurs à 15 minutes, par équipe", x=0.01, ha="left",
             fontsize=12, fontweight="semibold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

- Les distributions des deux équipes se superposent presque parfaitement.
- L'or, les sbires et les éliminations ont une forme en cloche légèrement étirée vers la droite
  (quelques parties très sanglantes ou très « farmées »).
- Les monstres de jungle forment des dents de scie, probablement parce que la jungle se nettoie par
  camps entiers de plusieurs monstres.
- Le niveau moyen ne prend que des valeurs multiples de 0,2 et reste très concentré (la moitié des
  équipes entre 9,0 et 9,4). Les parties anormales de la section 5.5 (niveau inférieur à 7) sont trop
  rares pour apparaître sur ce graphique.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [5, 12]})
width = 0.38
for ax, metric in zip(axes, ["EpicMonsters", "TowersLost"]):
    values = sorted(set(df[f"blue{metric}"]) | set(df[f"red{metric}"]))
    x = np.arange(len(values))
    for offset, team, color, label in ((-0.2, "blue", BLUE, "Équipe bleue"), (0.2, "red", RED, "Équipe rouge")):
        share = df[f"{team}{metric}"].value_counts(normalize=True).reindex(values, fill_value=0) * 100
        ax.bar(x + offset, share, width=width, color=color, label=label)
    ax.set_xticks(x, values)
    ax.grid(axis="x", visible=False)
    ax.set_title(TEAM_METRICS[metric])
axes[0].set_ylabel("Part des équipes (%)")
axes[1].legend(loc="upper right")
fig.tight_layout()
plt.show()

- Une équipe tue le plus souvent 1 ou 2 monstres épiques avant 15 minutes.
- La moitié des équipes n'a perdu aucune tour à 15 minutes ; en perdre 4 ou plus est rare (moins de 3 %
  des équipes) et correspond à des parties très déséquilibrées.

### 7.3 Répartition des classes (cible `blue_win`)

In [ ]:
from scipy.stats import binomtest

display(gdc.class_distribution(df, config))
n_games, n_blue = len(df), int(df["blue_win"].sum())
test = binomtest(n_blue, n_games, p=0.5)
print(f"Victoires bleues : {n_blue / n_games:.2%} ; test binomial (H0 : 50 %) : p-value = {test.pvalue:.3f}")

- Les classes sont **équilibrées** (environ 50,5 % / 49,5 %) : l'exactitude (accuracy) est une métrique
  lisible et aucun rééquilibrage n'est nécessaire.
- Le léger avantage du côté bleu, souvent observé dans le jeu, est statistiquement significatif
  (p-value inférieure à 0,05) mais faible.
- Prédire « l'équipe bleue gagne » à chaque fois donne environ 50,5 % d'exactitude : c'est la
  **référence à battre**.

### 7.4 Valeurs aberrantes : z-scores élevés

**Rappel.** Le z-score mesure l'écart d'une valeur à la moyenne, en nombre d'écarts-types :
z = (x - moyenne) / écart-type. Pour une loi normale, seulement 0,27 % des valeurs vérifient |z| > 3 :
on les considère donc comme aberrantes (ou atypiques).

**Limite.** La règle suppose une distribution à peu près symétrique, et la moyenne comme l'écart-type
sont eux-mêmes tirés par les valeurs extrêmes. On la compare donc à la méthode de l'écart interquartile
(IQR, méthode de Tukey), fondée sur les quartiles : est hors IQR toute valeur en dehors de
[Q1 - 1,5 × IQR ; Q3 + 1,5 × IQR].

In [ ]:
Z_THRESHOLD = config["datacard"]["z_threshold"]
outliers = pd.DataFrame({
    "z-score > 3": [int(gdc.zscore_mask(df[c], Z_THRESHOLD).sum()) for c in FEATURES],
    "hors IQR": [int(gdc.iqr_mask(df[c]).sum()) for c in FEATURES],
}, index=FEATURES)
outliers["z-score > 3 (%)"] = (outliers["z-score > 3"] / len(df) * 100).round(2)
flagged = pd.concat([gdc.zscore_mask(df[c], Z_THRESHOLD) for c in FEATURES], axis=1).any(axis=1)
print(f"Parties avec au moins une valeur |z| > 3 : {fr(flagged.sum())} ({flagged.mean():.1%})")
outliers

In [ ]:
order = outliers.sort_values("z-score > 3").index
y_pos = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9, 5.6))
ax.barh(y_pos + 0.2, outliers.loc[order, "z-score > 3"], height=0.38, color=VIOLET, label="|z| > 3")
ax.barh(y_pos - 0.2, outliers.loc[order, "hors IQR"], height=0.38, color=NEUTRAL, label="Hors IQR (Tukey)")
ax.set_yticks(y_pos, order)
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(thousands)
ax.set_xlabel("Nombre de valeurs signalées")
ax.legend(loc="lower right")
ax.set_title("Valeurs signalées comme aberrantes, par variable et par méthode")
fig.tight_layout()
plt.show()

In [ ]:
# Les parties les plus atypiques : plus grand |z| sur l'ensemble des variables
z_all = (df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()
most_atypical = z_all.abs().max(axis=1).sort_values(ascending=False).head(5)
cols = ["blueAvgLevel", "redAvgLevel", "blueTowersLost", "redTowersLost", "blueGold", "redGold", "blue_win"]
df.loc[most_atypical.index, cols].assign(z_max=most_atypical.round(1))

**Lecture**

- Pour les variables en cloche (or, sbires, éliminations), |z| > 3 concerne 0,4 à 0,6 % des valeurs,
  un peu plus que les 0,27 % d'une loi normale : les queues sont légèrement plus épaisses.
- Pour les **tours perdues**, très asymétriques (la moitié des équipes à 0), le z-score signale les
  équipes ayant perdu au moins 4 ou 5 tours (selon l'équipe), et l'IQR dès 3 tours. Ce sont pourtant
  des parties légitimes, simplement déséquilibrées : aucune des deux méthodes n'est adaptée telle
  quelle à ces comptages.
- Pour le **niveau moyen**, l'écart-type est si faible (0,4) qu'un niveau de 7,8 ou de 10,6 dépasse
  déjà |z| = 3.
- Les parties les plus atypiques (tableau ci-dessus) sont de deux sortes : des parties avec joueur
  absent (niveau moyen de 2,4 ou 5,4, jusqu'à |z| = 16) et des parties où une équipe a perdu ses
  11 tours avant 15 minutes. Le z-score fait bien ressortir les vraies anomalies, mais aussi les
  écrasements légitimes.

**Décision** : les z-scores élevés servent à **repérer** des parties à examiner, pas à les supprimer
automatiquement. Seules les erreurs avérées (doublons) sont retirées ; les parties anormales sont
signalées.

### 7.5 Relations avec la cible

Les deux équipes s'affrontent : ce qui compte, c'est l'**écart** entre elles. On construit des variables
d'écart (bleu moins rouge). Pour les tours, l'écart utile est « tours prises par l'équipe bleue moins
tours prises par l'équipe rouge », soit `redTowersLost - blueTowersLost`.

In [ ]:
diff = pd.DataFrame({
    "ecart_or": df["blueGold"] - df["redGold"],
    "ecart_sbires": df["blueMinionsKilled"] - df["redMinionsKilled"],
    "ecart_jungle": df["blueJungleMinionsKilled"] - df["redJungleMinionsKilled"],
    "ecart_niveau": df["blueAvgLevel"] - df["redAvgLevel"],
    "ecart_eliminations": df["blueChampKills"] - df["redChampKills"],
    "ecart_monstres_epiques": df["blueEpicMonsters"] - df["redEpicMonsters"],
    "ecart_tours": df["redTowersLost"] - df["blueTowersLost"],
})
corr = diff.assign(blue_win=df["blue_win"]).corr()
corr["blue_win"].drop("blue_win").sort_values(ascending=False).round(3).to_frame("corrélation avec blue_win")

In [ ]:
LABELS = {
    "ecart_or": "Or", "ecart_sbires": "Sbires", "ecart_jungle": "Jungle", "ecart_niveau": "Niveau",
    "ecart_eliminations": "Éliminations", "ecart_monstres_epiques": "Monstres épiques",
    "ecart_tours": "Tours prises", "blue_win": "Victoire bleue",
}
cmap = LinearSegmentedColormap.from_list("rouge_gris_bleu", [RED, "#f0efec", BLUE])
names = [LABELS[c] for c in corr.columns]
fig, ax = plt.subplots(figsize=(7.6, 6.2))
image = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=35, ha="right")
ax.set_yticks(range(len(names)), names)
ax.set_xticks(np.arange(-0.5, len(names)), minor=True)
ax.set_yticks(np.arange(-0.5, len(names)), minor=True)
ax.grid(False)
ax.grid(which="minor", color=SURFACE, linewidth=2)  # liseré entre les cellules
ax.tick_params(which="both", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
for i in range(len(names)):
    for j in range(len(names)):
        value = corr.values[i, j]
        ax.text(j, i, f"{value:.2f}".replace(".", ","), ha="center", va="center", fontsize=8.5,
                color="white" if abs(value) > 0.6 else INK)
colorbar = fig.colorbar(image, ax=ax, shrink=0.8)
colorbar.outline.set_visible(False)
ax.set_title("Corrélations entre les écarts (bleu moins rouge) et la victoire bleue")
fig.tight_layout()
plt.show()

In [ ]:
def win_rate_by_gap(gap, step, min_games=200):
    """Taux de victoire bleue par classe d'écart (classes centrées sur les multiples de step)."""
    classes = (gap / step).round().astype(int) * step
    stats = df["blue_win"].groupby(classes).agg(["mean", "size"])
    return stats[stats["size"] >= min_games]

CURVES = [("ecart_or", 500, "Écart d'or (pièces)"),
          ("ecart_eliminations", 1, "Écart d'éliminations"),
          ("ecart_tours", 1, "Écart de tours prises")]
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (col, step, xlabel) in zip(axes, CURVES):
    curve = win_rate_by_gap(diff[col], step)
    ax.axhline(50, color=AXIS, linewidth=1)
    ax.plot(curve.index, curve["mean"] * 100, color=BLUE, marker="o", markersize=5,
            markeredgecolor=SURFACE, markeredgewidth=1.5)
    ax.set_xlabel(xlabel)
    if col == "ecart_or":
        ax.xaxis.set_major_locator(MultipleLocator(5000))
        ax.xaxis.set_major_formatter(thousands)
axes[0].set_ylabel("Victoires de l'équipe bleue (%)")
axes[0].set_ylim(0, 100)
fig.suptitle("Taux de victoire de l'équipe bleue selon l'écart à 15 minutes (classes d'au moins 200 parties)",
             x=0.01, ha="left", fontsize=12, fontweight="semibold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

gold_curve = win_rate_by_gap(diff["ecart_or"], 500)
for gap in (0, 1000, 2000, 4000):
    print(f"Écart d'or de +{fr(gap)} (à 250 près) : {gold_curve.loc[gap, 'mean']:.0%} de victoires bleues")

- L'**écart d'or** est la variable la plus liée à la victoire (corrélation de 0,61), devant les
  éliminations (0,58) et le niveau (0,56).
- Ces variables sont aussi **fortement corrélées entre elles** : une élimination rapporte de l'or et de
  l'expérience. Un modèle linéaire s'en accommode pour prédire, mais l'interprétation de ses
  coefficients demandera de la prudence (projet 2).
- Le taux de victoire suit une **courbe en S** : environ 50 % à écart d'or nul, 60 % à +1 000,
  70 % à +2 000 et 85 % à +4 000 pièces d'or. C'est exactement la forme que modélise une régression
  logistique.

## 8. Dictionnaire des données

Le dictionnaire décrit le batch **tel qu'il est livré** par OpenML (avant nettoyage), car c'est ce que
documente la datacard. Il est calculé par la fonction `build_data_dictionary` du script
`generate_datacard.py` : le notebook et la datacard affichent exactement les mêmes chiffres. Les
descriptions et unités viennent de `datacard_config.json` (rédigé à la main).

| Colonne du dictionnaire | Contenu |
|---|---|
| `colonne`, `description`, `unite` | Nom, sens métier et unité |
| `role` | cible, identifiant, technique ou variable |
| `type_stocke`, `nature` | Type pandas et nature statistique (binaire, constante, quantitative discrète ou continue) |
| `n_valeurs_distinctes` | Nombre de valeurs différentes |
| `min`, `max` | Intervalle des valeurs |
| `moyenne`, `mediane`, `q1`, `q3`, `ecart_type` | Tendance centrale, quartiles et dispersion |
| `n_manquants`, `pct_manquants` | Valeurs manquantes |
| `n_z_eleve`, `pct_z_eleve` | Valeurs dont le z-score dépasse 3 en valeur absolue |
| `n_hors_iqr` | Valeurs hors de [Q1 - 1,5 × IQR ; Q3 + 1,5 × IQR] |

Pour les identifiants et la colonne technique, seuls le minimum et le maximum ont un sens ; les
aberrants ne sont pas calculés pour les variables binaires ou constantes.

In [ ]:
dictionary = gdc.build_data_dictionary(raw, config)
dictionary.to_csv(OUTPUT_DIR / "data_dictionary.csv", index=False)

stats_cols = ["role", "type_stocke", "nature", "unite", "n_valeurs_distinctes", "min", "max", "moyenne",
              "mediane", "q1", "q3", "ecart_type", "n_manquants", "n_z_eleve", "pct_z_eleve", "n_hors_iqr"]
with pd.option_context("display.float_format", lambda v: f"{v:,.2f}".replace(",", " ")):
    display(dictionary.set_index("colonne")[stats_cols])

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(dictionary.set_index("colonne")[["description"]])

Les indicateurs qui portent sur le jeu entier plutôt que sur une colonne (répartition des classes,
doublons) complètent le dictionnaire :

In [ ]:
quality = gdc.quality_report(raw, config)
display(gdc.class_distribution(raw, config))
pd.Series({
    "Valeurs manquantes (total)": quality["manquants_total"],
    "Doublons sur toutes les colonnes": quality["doublons_complets"],
    "Doublons en ignorant Unnamed:_0": quality["doublons_hors_techniques"],
    "matchId en double": quality["doublons_identifiants"]["matchId"],
    "Colonnes constantes": ", ".join(quality["colonnes_constantes"]),
    "Lignes avec au moins un |z| > 3": quality["lignes_avec_z_eleve"],
}, name="valeur").to_frame()

## 9. Projets de Machine Learning

Trois idées, de la plus accessible à la plus difficile. Pour les deux projets réalistes, un test de
faisabilité rapide (quelques lignes, sans réglage) vérifie que l'idée tient sur ces données.

### Projet 1 (réaliste) : probabilité de victoire en direct à 15 minutes

**Idée.** Afficher pendant une partie, par exemple en overlay de stream ou lors de l'analyse d'une VOD,
la probabilité de victoire de chaque équipe calculée à 15:00.

**Pourquoi ce projet m'intéresse.** C'est un outil concret pour un joueur ou un streamer : il rend
visible le poids d'un début de partie et sert de support d'analyse.

| | |
|---|---|
| Tâche | Classification binaire probabiliste (cible `blue_win`) |
| Variables | Écarts bleu moins rouge (or, sbires, jungle, niveau, éliminations, monstres épiques, tours) |
| Métriques | Log loss et score de Brier (qualité des probabilités), AUC, exactitude, calibration |
| Référence à battre | Classe majoritaire : environ 50,5 % d'exactitude |
| Contrainte du direct | L'API locale du client de jeu (Live Client Data API) donne pour tous les joueurs les éliminations, le score de sbires, le niveau et les objectifs, mais **l'or courant seulement pour le joueur actif** : il faut une version du modèle sans l'or d'équipe |

**Démarche.**
1. Partir des écarts de la section 7.5, sans les doublons ni les parties suspectes.
2. Comparer une régression logistique (interprétable, en général bien calibrée) à un modèle de gradient boosting.
3. Évaluer par validation croisée et sur un découpage temporel (parties les plus récentes en test).
4. Vérifier la calibration : une probabilité annoncée de 70 % doit se réaliser environ 7 fois sur 10.
5. Entraîner la version « direct » sans l'or et mesurer ce qu'elle perd.
6. Brancher le modèle sur la Live Client Data API pour l'affichage.

**Risques.** Plafond de performance (la partie continue après 15 minutes) et écart entre le jeu de 2020
et le jeu actuel (voir projet 3).

**Test de faisabilité**

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

keep = ~df["partie_suspecte"]
X = diff[keep].assign(ecart_cs_total=lambda t: t["ecart_sbires"] + t["ecart_jungle"])
y = df.loc[keep, "blue_win"]

ALL = ["ecart_or", "ecart_sbires", "ecart_jungle", "ecart_niveau",
       "ecart_eliminations", "ecart_monstres_epiques", "ecart_tours"]
LIVE = ["ecart_cs_total", "ecart_niveau", "ecart_eliminations", "ecart_monstres_epiques", "ecart_tours"]


def logistic():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


candidates = {
    "Classe majoritaire (référence)": (DummyClassifier(strategy="prior"), ALL),
    "Logistique, écart d'or seul": (logistic(), ["ecart_or"]),
    "Logistique, toutes les variables": (logistic(), ALL),
    "Logistique, variables du direct (sans or)": (logistic(), LIVE),
    "Gradient boosting, toutes les variables": (HistGradientBoostingClassifier(random_state=RANDOM_STATE), ALL),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"exactitude": "accuracy", "AUC": "roc_auc", "Brier": "neg_brier_score", "log loss": "neg_log_loss"}
results = {}
for name, (model, columns) in candidates.items():
    scores = cross_validate(model, X[columns], y, cv=cv, scoring=scoring)
    results[name] = {metric: abs(scores[f"test_{metric}"].mean()) for metric in scoring}
with pd.option_context("display.precision", 3):
    display(pd.DataFrame(results).T)

In [ ]:
# Découpage temporel : les identifiants Riot croissent avec le temps, on teste sur les 20 % les plus récents
rank = df.loc[keep, "matchId"].rank(method="first")
train, test = rank <= rank.quantile(0.8), rank > rank.quantile(0.8)
model = logistic().fit(X.loc[train, ALL], y[train])
proba = model.predict_proba(X.loc[test, ALL])[:, 1]
print(f"Découpage temporel : exactitude {accuracy_score(y[test], proba > 0.5):.3f}, "
      f"AUC {roc_auc_score(y[test], proba):.3f}")

observed, predicted = calibration_curve(y[test], proba, n_bins=5)
pd.DataFrame({"probabilité prédite (moyenne)": predicted, "victoires bleues observées": observed}).round(3)

**Bilan de faisabilité.** Avec toutes les variables, la régression logistique atteint environ 79 %
d'exactitude (AUC proche de 0,87), loin devant la référence. Le gradient boosting ne fait pas mieux :
la relation est presque linéaire dans les écarts, et la limite vient de l'information disponible à
15 minutes plutôt que du modèle. La version **sans or** perd moins d'un point : le projet est faisable
en direct. Les scores tiennent sur les parties les plus récentes et les probabilités sont bien
calibrées (prédit et observé très proches), ce qui est indispensable pour un affichage en pourcentage.
C'est nettement mieux que le plafond d'environ 70 % annoncé à 10 minutes, ce qui est logique : à
15 minutes, les écarts entre équipes sont plus marqués.

### Projet 2 (réaliste) : combien vaut un objectif ?

**Idée.** Exprimer la valeur d'une élimination, d'un monstre épique, d'une tour ou d'un sbire dans une
unité commune, l'**or équivalent** : « cette action change autant la probabilité de victoire qu'un
avantage de X pièces d'or ».

**Pourquoi ce projet m'intéresse.** En partie, la question « faut-il échanger une tour contre un
objectif ? » revient sans cesse. Un modèle interprétable donne des ordres de grandeur chiffrés.

| | |
|---|---|
| Tâche | Modèle interprétable (régression logistique), puis lecture des coefficients |
| Variables | Écarts d'éliminations, de monstres épiques, de tours, de sbires et de monstres de jungle |
| Livrable | Un tableau « valeur en or équivalent » avec intervalles de confiance |

**Point de méthode.** L'or est une **conséquence** des éliminations et des objectifs (variable
médiatrice). Si on le met dans le modèle, on mesure l'effet d'une action *à or égal*, ce qui efface une
grande partie de sa valeur. On estime donc un modèle **sans l'or** (effet total des actions), puis on
convertit chaque coefficient en or grâce à la pente du modèle « écart d'or seul ».

**Démarche.** 1. Modèle sans or ; 2. conversion en or équivalent ; 3. intervalles de confiance par
bootstrap ; 4. stabilité selon le type de partie (serrée ou non) ; 5. comparaison avec les vraies
récompenses du jeu.

**Limites.** Corrélation n'est pas causalité (une équipe en avance prend aussi plus d'objectifs) ;
dragons et Hérauts sont mélangés dans la même colonne.

**Test de faisabilité**

In [ ]:
ACTIONS = ["ecart_eliminations", "ecart_monstres_epiques", "ecart_tours", "ecart_sbires", "ecart_jungle"]


def raw_coefficients(pipeline):
    """Coefficients par unité d'origine : on annule la standardisation (coef / écart-type)."""
    scaler, logreg = pipeline[0], pipeline[-1]
    return logreg.coef_[0] / scaler.scale_


actions_model = logistic().fit(X[ACTIONS], y)
gold_slope = raw_coefficients(logistic().fit(X[["ecart_or"]], y))[0]  # log-cote par pièce d'or d'écart
gold_equivalent = pd.Series(raw_coefficients(actions_model) / gold_slope, index=ACTIONS)
gold_equivalent.round(0).rename("valeur d'une unité, en or équivalent").to_frame()

**Bilan de faisabilité.** Les ordres de grandeur sont crédibles : une élimination, une tour ou un
monstre épique valent chacun de l'ordre de 400 pièces d'or, un sbire de voie environ 20 et un monstre
de jungle environ 30. Bon signe : la valeur d'un sbire de voie retombe sur l'or qu'il rapporte
réellement en début de partie (environ 20 pièces en moyenne), et celle d'une élimination est proche
de sa prime (300 pièces, plus l'or des assistances). Le projet est donc réaliste ; le travail
consistera à fiabiliser ces chiffres (intervalles de confiance, colinéarité, séparation dragons et
Héraut avec des données corrigées).

### Projet 3 (difficile) : adapter le modèle au jeu actuel et à un autre niveau

**Idée.** Faire fonctionner le prédicteur du projet 1 sur des parties d'aujourd'hui (saison 2026) et
d'un autre niveau, par exemple Silver, avec mes propres parties comme jeu de test.

**Pourquoi c'est difficile.**

- **Nouvelle collecte obligatoire** : récupérer des chronologies de parties récentes via l'API Match v5
  de Riot (clé d'API, limites de requêtes, choix des joueurs), en corrigeant au passage les défauts de
  ce jeu (dragons, tours, Hérauts séparés).
- **Le jeu a changé** : depuis 2024, les Larves du Néant remplacent le premier Héraut, et d'autres
  objectifs et objets ont évolué. Certaines variables n'ont plus le même sens : le **schéma** change.
- **Changement de distribution** (*domain shift*) : hypothèse à tester, un avantage à 15 minutes se
  convertit peut-être moins souvent en victoire à un niveau plus bas (plus d'erreurs, plus de
  retournements). Un modèle entraîné en Diamant 2020 risque d'être trop confiant.
- **Peu de données étiquetées** dans le domaine visé au départ : quelques centaines de parties
  personnelles ne suffisent pas pour ré-entraîner, il faut combiner transfert et recalibration.
- **Évaluation honnête** : séparer les parties par joueur et par date pour éviter les fuites
  d'information.

**Démarche envisagée.** 1. Pipeline de collecte (Match v5 et sa chronologie) ; 2. reconstruction des
variables à 15 minutes ; 3. mesure de l'écart de distribution entre 2020 et 2026 (le rôle de la section
« Évolution » de la datacard entre deux batchs) ; 4. recalibration du modèle du projet 1 sur un petit
échantillon récent ; 5. ré-entraînement si nécessaire, puis comparaison.

## 10. Datacard

La datacard est découpée en trois fichiers, pour séparer ce qui est écrit à la main de ce qui est calculé :

| Fichier | Rédaction | Contenu |
|---|---|---|
| `datacard_header.md` | À la main | Résumé, provenance, collecte, modèle du domaine, usages, limites, éthique |
| `datacard_config.json` | À la main | Rôles des colonnes, descriptions, unités, règles métier |
| `generate_datacard.py` | Script | Calcule la section 8 de la datacard (empreinte, dictionnaire, classes, qualité, règles, évolution) et assemble `DATACARD.md` |

```
datacard_header.md   (à la main)  --+
datacard_config.json (à la main)  --+-->  generate_datacard.py  -->  DATACARD.md
data/raw/<batch>.csv (données)    --+                           -->  outputs/data_dictionary.csv
                                                                -->  outputs/datacard_stats.json
```

`outputs/datacard_stats.json` garde un résumé chiffré du batch : à la génération suivante, le script
compare le nouveau batch à celui-ci (lignes, colonnes, types, moyennes, part de chaque classe) et
l'écrit dans la section « Évolution ». Une empreinte SHA-256 du contenu permet de savoir si les données
ont réellement changé.

### 10.1 Génération de la datacard

In [ ]:
batch_file = RAW_ARFF.relative_to(PROJECT_DIR)  # équivalent : python generate_datacard.py --source <fichier>
result = gdc.generate_datacard(batch_file)
for key in ("output_path", "dictionary_path", "snapshot_path"):
    print(Path(result[key]).relative_to(PROJECT_DIR))
display(Markdown(result["generated_markdown"]))

### 10.2 Simulation de l'arrivée d'un nouveau batch

Pour vérifier que la datacard se met à jour, on simule deux livraisons successives : un premier batch
avec les 80 % de parties les plus anciennes (selon `matchId`), puis le batch complet. Les fichiers de
cette démonstration vont dans `outputs/demo_batch/` pour ne pas toucher à la vraie datacard.

In [ ]:
demo_dir = OUTPUT_DIR / "demo_batch"
demo_dir.mkdir(exist_ok=True)
demo_paths = {
    "output_path": demo_dir / "DATACARD_demo.md",
    "dictionary_path": demo_dir / "data_dictionary_demo.csv",
    "snapshot_path": demo_dir / "datacard_stats_demo.json",
}
demo_paths["snapshot_path"].unlink(missing_ok=True)  # la démonstration repart de zéro

batch_1 = raw[raw["matchId"] <= raw["matchId"].quantile(0.8)]
gdc.generate_datacard(df=batch_1, source_label="batch 1 (80 % des parties, les plus anciennes)", **demo_paths)
batch_2 = gdc.generate_datacard(df=raw, source_label="batch 2 (toutes les parties)", **demo_paths)

evolution = batch_2["generated_markdown"].split("### 8.7")[1].split(gdc.AUTO_END)[0]
display(Markdown("#### Section 8.7 produite pour le batch 2" + evolution.split("\n", 1)[1]))

**En ligne de commande**, pour un vrai nouveau batch :

```bash
python generate_datacard.py --source data/raw/<nouveau_batch>.csv
```

Le script réécrit la section 8 de `DATACARD.md`, met à jour `outputs/data_dictionary.csv` et compare
automatiquement au batch précédent. Il est générique : un nouveau batch avec une colonne en plus est
signalé (« colonnes absentes de la configuration »), une règle qui cite une colonne disparue devient
« non évaluable ». L'entête n'est jamais modifiée par le script.

## 11. Bilan

**Ce que l'exploration a appris**

1. **Collecte** : bien documentée sur l'essentiel (API Riot, NA1, Diamant, 15 minutes), incomplète sur
   la période exacte, la version du jeu et l'échantillonnage.
2. **Sens des colonnes** : sur les quatre familles de colonnes d'actions (éliminations, Hérauts,
   dragons, tours), trois ne mesurent pas ce que leur nom annonce. Dragons vides, « Hérauts » qui
   comptent très probablement tous les monstres épiques, « tours détruites » qui comptent les tours
   perdues. Sans modèle du domaine, ces erreurs passaient inaperçues et faussaient toute interprétation.
3. **Qualité technique** : bonne. Aucune valeur manquante, 19 doublons, une trentaine de parties
   anormales, des types à corriger (entiers stockés en flottant, attention aux types non signés).
4. **Signal** : classes équilibrées ; l'écart d'or concentre l'essentiel de l'information, et un modèle
   simple atteint environ 79 % d'exactitude à 15 minutes.

**Suite** : commencer par le projet 1 (modèle calibré, version sans or), qui sert de base aux projets 2
et 3, et relancer `generate_datacard.py` à chaque nouveau batch.